# Optimal Mixing of Passive Scalars — Python Simulation

This notebook reproduces the numerical experiments from:

> Gautam Iyer, *Mix Norms and the Rate of Decay for the Passive Scalar Equation*  
> (companion code: [math.cmu.edu/~gautam/research/201208-mix-bounds](https://www.math.cmu.edu/~gautam/research/201208-mix-bounds/))

The simulation solves the **passive scalar transport equation**:

$$\partial_t \theta + (u \cdot \nabla)\theta = 0$$

using the **optimal mixing velocity** of Lin–Thiffeault–Doering:

$$v = -\Delta^{-1} P(\theta \nabla \Delta^{-1} \theta), \qquad u = F \frac{v}{\|\nabla v\|_{L^2}}$$

where $P$ is the Leray projection and $F$ is the enstrophy constraint.

The **$H^{-1}$ mix norm** quantifies mixing:

$$\|\theta\|_{H^{-1}}^2 = \sum_{k \neq 0} \frac{|\hat{\theta}(k)|^2}{4\pi^2 |k|^2}$$

| Component | Approach |
|---|---|
| Spatial discretisation | Pseudo-spectral (2D FFT) on $[0,1]^2$ |
| Time integration | RK45 (`scipy.integrate.solve_ivp`) |
| Resolution check | Stop when $L^p$ norms drift $> 10^{-3}$ |

## Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), 'python_code'))

import numpy as np
import matplotlib.pyplot as plt
from mixing import (
    build_operators,
    idata_sin, idata_diag, idata_strip, idata_trigpoly,
    run_simulation,
    plot_mix_norm, plot_lp_norms, plot_mixing_rate, plot_snapshots,
    verify,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## Spectral operators

Derivatives in Fourier space:

$$\widehat{\partial_x f}(k) = 2\pi i k_x \hat{f}(k), \qquad\widehat{\Delta^{-1} f}(k) = \frac{\hat{f}(k)}{-(2\pi)^2|k|^2}$$

For even $N$ the Nyquist mode is zeroed in first-derivative operators.

In [ ]:
N = 64           # spectral resolution (power of 2)
F = 1.0          # enstrophy constraint

ops = build_operators(N)
print(f"Grid: {N}x{N} modes,  dx = {ops['dx']:.4f}")
print(f"LAP_INV shape: {ops['LAP_INV'].shape}")

## Initial conditions

Four families of initial data, all $L^2$-normalised and supported in $[0,a]^2$.
Parameter $a \in (0,1)$ controls the support size.

In [ ]:
a = 0.5

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (fn, name) in zip(axes, [
    (idata_sin,      r'sin(2$\pi$ x/a) sin(2$\pi$ y/a)'),
    (idata_diag,     'Diagonal'),
    (idata_strip,    'Strip'),
    (idata_trigpoly, 'Trig poly'),
]):
    theta0 = fn(a, ops)
    im = ax.imshow(theta0, origin='lower', extent=[0,1,0,1], cmap='RdBu_r', aspect='equal')
    ax.set_title(name, fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f'Initial conditions (a = {a})', fontsize=11)
plt.tight_layout()
plt.show()

## Single-run simulation

Run for $a = 0.5$ and observe spatial mixing and norm evolution.

In [ ]:
a     = 0.5
t_eval = np.arange(0, 5.05, 0.05)

print(f'Running: N={N}, F={F}, a={a}')
res = run_simulation(a, idata_sin, ops, F=F, t_eval=t_eval)
print(f'  Stopped at t = {res["t"][-1]:.2f}')
print(f'  L2 drift:     {np.max(np.abs(res["norm_l2"] - 1)):.2e}')
print(f'  H-1 at t_end: {res["norm_hm1"][-1]:.4f}')

In [ ]:
fig = plot_snapshots(res['theta'], res['t'], ops, n_frames=6,
                     title=f'Scalar field, a={a}')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(res['t'], np.log(res['norm_hm1']), 'b-')
axes[0].set_xlabel('t')
axes[0].set_ylabel('log ||theta||_{H-1}')
axes[0].set_title(f'H-1 mix norm (log scale), a={a}')

axes[1].plot(res['t'], res['norm_l2'], 'r', label='L2')
axes[1].plot(res['t'], res['norm_l4'], 'g', label='L4')
axes[1].plot(res['t'], res['norm_l8'], 'b', label='L8')
axes[1].legend()
axes[1].set_xlabel('t')
axes[1].set_title('Lp norms (conservation check)')
plt.tight_layout()
plt.show()

## Multi-scale sweep

Vary $a$ to compare mixing rates across scales.
The slope of $\log \|\theta\|_{H^{-1}}$ gives the mixing rate.

In [ ]:
# Full MATLAB range: a_range = np.arange(0.5, 1.0, 1/16)
a_range = [0.50, 0.625, 0.75, 0.875]   # smaller set for speed
t_eval  = np.arange(0, 5.05, 0.05)

print(f'Running {len(a_range)} simulations on N={N} grid ...')
results = []
for a in a_range:
    print(f'  a = {a:.4f}', end='', flush=True)
    res = run_simulation(a, idata_sin, ops, F=F, t_eval=t_eval)
    print(f'  stopped at t={res["t"][-1]:.2f},  H-1={res["norm_hm1"][-1]:.4f}')
    results.append(res)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_mix_norm(results, a_range, ax=axes[0])
plot_lp_norms(results, ax=axes[1])
plot_mixing_rate(results, a_range, ax=axes[2])
plt.tight_layout()
plt.show()

## Verification

Sanity checks on a small $N=32$ grid:
- **L² conservation**: drift $< 5 \times 10^{-4}$
- **H⁻¹ decay**: ratio $< 1$ (mixing has occurred)

In [ ]:
_, ok = verify(N=32, F=1.0, a=0.5, t_end=0.5)

## Algorithm summary

### Pseudo-spectral computation of $u$

At each RK45 stage:

1. $g = \theta \nabla \Delta^{-1}\theta$  — products computed in physical space
2. $Pg = g - \nabla \Delta^{-1}(\nabla \cdot g)$  — Leray projection in Fourier space
3. $v = -\Delta^{-1} Pg$  — velocity stream function
4. $u = F v / \|\nabla v\|_{L^2}$  — rescale to enstrophy constraint
5. $\partial_t \hat{\theta} = -\widehat{u \cdot \nabla \theta}$  — advection in Fourier space

### MATLAB correspondence

| MATLAB file | Python function |
|---|---|
| `gen_figures.m` | `run_simulation()` |
| `convection_hat.m` | `make_convection_hat()` |
| `fn_norm.m` | `compute_norms()` |
| `res_check.m` | `make_res_check()` |
| `idata_sin/diag/strip/trigpoly` | same names in `mixing.py` |
| `ode45(...)` | `solve_ivp(..., method='RK45')` |
| `ifft2(..., 'symmetric')` | `np.real(np.fft.ifft2(...))` |